# Add Market Values to Player Database

Adds season start and season end market values from Transfermarkt.

**Run this AFTER all other data has been added to the database.**

In [23]:
import pandas as pd
import numpy as np
import os
import unicodedata
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [24]:
RAW_DIR = os.path.join('..', 'data', 'raw', 'player_scores')
DB_PATH = os.path.join('..', 'data', 'processed', 'player_db.csv')

# Use FILTERED Transfermarkt data files (run filter_transfermarkt_data.ipynb first!)
PLAYERS_FILE = os.path.join(RAW_DIR, 'players_big5_filtered.csv')
VALUES_FILE = os.path.join(RAW_DIR, 'player_valuations_big5_filtered.csv')

# Season date ranges (approximate)
SEASON_DATES = {
    '2020-2021': {'start': '2020-09-01', 'end': '2021-05-31'},
    '2021-2022': {'start': '2021-08-01', 'end': '2022-05-31'},
    '2022-2023': {'start': '2022-08-01', 'end': '2023-05-31'},
    '2023-2024': {'start': '2023-08-01', 'end': '2024-05-31'},
    '2024-2025': {'start': '2024-08-01', 'end': '2025-05-31'},
}

In [25]:
def normalise_name(name):
    """Normalize names for matching."""
    if pd.isna(name):
        return ''
    name = str(name).strip().lower()
    name = unicodedata.normalize('NFD', name)
    return ''.join(c for c in name if unicodedata.category(c) != 'Mn')

## 1. Load Master Database

In [26]:
if not os.path.exists(DB_PATH):
    print(f"❌ Database not found: {DB_PATH}")
else:
    master = pd.read_csv(DB_PATH)
    master['name_clean'] = master['name'].apply(normalise_name)
    print(f"✓ Loaded database: {len(master):,} rows")
    display(master.head())

✓ Loaded database: 13,636 rows


,rank,name,Nation,position,club,league_comp,Age,Born,appearances,Starts,...,team,xG,xA,xG90,xA90,xG90xA90,NPxG90xA90,xGChain90,xGBuildup90,name_clean
0,2700,Aaron Ciammaglichella,it ITA,MF,Torino,it Serie A,19.0,2005.0,1,0,...,Torino,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,aaron ciammaglichella
1,1713,Aaron Connolly,ie IRL,FW/MF,Brighton,eng Premier League,20.0,2000.0,17,9,...,Brighton,4.46,0.16,0.50,0.02,0.52,0.52,0.54,0.02,aaron connolly
2,2318,Aaron Connolly,ie IRL,FW,Brighton,eng Premier League,21.0,2000.0,4,1,...,Brighton,0.60,0.38,0.35,0.23,0.58,0.58,0.60,0.02,aaron connolly
3,67,Aaron Cresswell,eng ENG,DF,West Ham United,eng Premier League,30.0,1989.0,36,36,...,West Ham,0.88,7.39,0.03,0.21,0.23,0.23,0.30,0.24,aaron cresswell
4,269,Aaron Cresswell,eng ENG,DF,West Ham United,eng Premier League,31.0,1989.0,31,31,...,West Ham,0.91,3.67,0.03,0.12,0.15,0.15,0.35,0.32,aaron cresswell


## Manual Name Mappings for Transfermarkt

Apply same manual mappings as used for Understat data.

In [27]:
# Manual name mappings - same as used for Understat
# This ensures consistency across all data sources

MANUAL_NAME_MAPPINGS = {
    # General mappings (any season/team)
    'obite ndicka': 'eric ndicka',
    'pascu': 'jorge pascual',
    'przemyslaw placheta': 'przemyslaw placheta',
    'simeon nwankwo': 'simy',
    'tachi': 'alberto rodriguez',
}

# Context-specific mappings (specific team/season combinations)
CONTEXT_MAPPINGS = [
    ('rodri', '2020-2021', 'betis', 'rodri sanchez'),
    ('rodri', '2021-2022', 'betis', 'rodri sanchez'),
    ('rodri', '2022-2023', 'betis', 'rodri sanchez'),
    ('rodri', '2023-2024', 'betis', 'rodri sanchez'),
    ('rodri', '2024-2025', 'betis', 'rodri sanchez'),
    ('ridle baku', '2024-2025', 'leipzig', 'bote baku'),
    ('terem moffi', '2022-2023', 'lorient', 'terem igobor moffi'),
]

# Apply manual mappings
if 'master' in locals():
    print("Applying manual name mappings...\n")
    
    # Simple mappings
    for old_name, new_name in MANUAL_NAME_MAPPINGS.items():
        mask = master['name_clean'] == old_name
        count = mask.sum()
        if count > 0:
            master.loc[mask, 'name_clean'] = new_name
            print(f"  ✓ Mapped '{old_name}' → '{new_name}' ({count} rows)")
    
    # Context-specific mappings
    for name, season, club_keyword, mapped_name in CONTEXT_MAPPINGS:
        mask = (
            (master['name_clean'] == name) &
            (master['season'] == season) &
            (master['club'].str.lower().str.contains(club_keyword, na=False))
        )
        count = mask.sum()
        if count > 0:
            master.loc[mask, 'name_clean'] = mapped_name
            print(f"  ✓ Mapped '{name}' → '{mapped_name}' for {season} {club_keyword} ({count} rows)")
    
    print("\nManual mappings complete.")
else:
    print("⚠️  master not loaded yet")

Applying manual name mappings...

  ✓ Mapped 'pascu' → 'jorge pascual' (2 rows)
  ✓ Mapped 'simeon nwankwo' → 'simy' (3 rows)
  ✓ Mapped 'tachi' → 'alberto rodriguez' (2 rows)
  ✓ Mapped 'rodri' → 'rodri sanchez' for 2020-2021 betis (1 rows)
  ✓ Mapped 'rodri' → 'rodri sanchez' for 2021-2022 betis (1 rows)
  ✓ Mapped 'rodri' → 'rodri sanchez' for 2022-2023 betis (1 rows)
  ✓ Mapped 'rodri' → 'rodri sanchez' for 2023-2024 betis (1 rows)
  ✓ Mapped 'rodri' → 'rodri sanchez' for 2024-2025 betis (1 rows)
  ✓ Mapped 'ridle baku' → 'bote baku' for 2024-2025 leipzig (1 rows)
  ✓ Mapped 'terem moffi' → 'terem igobor moffi' for 2022-2023 lorient (1 rows)

Manual mappings complete.


## 2. Load Transfermarkt Data

In [28]:
# Load players metadata
if not os.path.exists(PLAYERS_FILE):
    print(f"❌ Players file not found: {PLAYERS_FILE}")
    tm_players = pd.DataFrame()
else:
    tm_players = pd.read_csv(PLAYERS_FILE)
    
    # Create normalized name for matching
    tm_players['name_clean'] = tm_players['name'].apply(normalise_name)
    
    print(f"✓ Loaded {len(tm_players):,} Transfermarkt players")
    display(tm_players.head())

✓ Loaded 8,692 Transfermarkt players


,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,height_in_cm,contract_expiration_date,agent_name,image_url,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur,name_clean
0,215,Roque,Santa Cruz,Roque Santa Cruz,2015,1084,roque-santa-cruz,Paraguay,Asunción,Paraguay,...,193.0,2023-12-31 00:00:00,NaN,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/roque-santa-cr...,ES1,Málaga CF,250000.0,12000000.0,roque santa cruz
1,532,Claudio,Pizarro,Claudio Pizarro,2019,86,claudio-pizarro,Peru,Callao,Peru,...,184.0,NaN,NaN,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/claudio-pizarr...,L1,Sportverein Werder Bremen von 1899,400000.0,12000000.0,claudio pizarro
2,1586,Christian,Schulz,Christian Schulz,2015,42,christian-schulz,Germany,Bassum,Germany,...,185.0,NaN,ARP Sportmarketing,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/christian-schu...,L1,Hannover 96,100000.0,4500000.0,christian schulz
3,2050,Nelson,Valdez,Nelson Valdez,2014,24,nelson-valdez,Paraguay,Caaguazú,Paraguay,...,179.0,NaN,GG11,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/nelson-valdez/...,L1,Eintracht Frankfurt Fußball AG,100000.0,5000000.0,nelson valdez
4,2421,Michael,Rensing,Michael Rensing,2019,38,michael-rensing,Germany,Lingen,Germany,...,190.0,NaN,Sports360 GmbH,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/michael-rensin...,L1,Fortuna Düsseldorf,300000.0,4500000.0,michael rensing


In [29]:
# Load market value history
if not os.path.exists(VALUES_FILE):
    print(f"❌ Valuations file not found: {VALUES_FILE}")
    tm_values = pd.DataFrame()
else:
    tm_values = pd.read_csv(VALUES_FILE)
    tm_values['date'] = pd.to_datetime(tm_values['date'])
    
    print(f"✓ Loaded {len(tm_values):,} market value records")
    display(tm_values.head())

✓ Loaded 73,036 market value records


,player_id,date,market_value_in_eur,current_club_id,player_club_domestic_competition_id
0,5794,2020-01-01,500000,3368,ES1
1,21972,2020-01-01,200000,607,IT1
2,34409,2020-01-01,1000000,873,GB1
3,240803,2020-01-01,450000,276,IT1
4,443710,2020-01-01,200000,79,L1


## 3. Match Players by Name

In [30]:
if not tm_players.empty and 'master' in locals():
    import re
    
    print("Matching players to Transfermarkt...\n")
    
    # Helper function to split names into words
    def split_name_words(name):
        """Split name on spaces, apostrophes, and hyphens for better matching."""
        if pd.isna(name):
            return set()
        words = re.split(r"[\s'\-\.]+", str(name).lower())
        return set(w for w in words if len(w) > 1)
    
    # Prepare TM data
    tm_players['dob'] = pd.to_datetime(tm_players['date_of_birth'], errors='coerce')
    tm_players['birth_year'] = tm_players['dob'].dt.year
    
    # Calculate expected birth year in master
    def calculate_birth_year(row):
        if pd.isna(row['Age']) or pd.isna(row['season']):
            return None
        season_year = int(row['season'].split('-')[0])
        return season_year - int(row['Age'])
    
    master['birth_year_expected'] = master.apply(calculate_birth_year, axis=1)
    
    print("Stage 1: Exact name + birth year matching...")
    
    # Create a mapping of (name_clean, birth_year) -> player_id
    # Only include TM players where birth_year is valid
    tm_valid = tm_players[tm_players['birth_year'].notna()].copy()
    tm_lookup = tm_valid.groupby(['name_clean', 'birth_year']).apply(
        lambda x: x.nlargest(1, 'highest_market_value_in_eur')['player_id'].iloc[0]
    ).to_dict()
    
    # Match with ±1 year tolerance
    def find_player_id(row):
        name = row['name_clean']
        birth_year = row['birth_year_expected']
        
        if pd.isna(birth_year):
            return None
        
        # Try exact birth year first
        key = (name, birth_year)
        if key in tm_lookup:
            return tm_lookup[key]
        
        # Try ±1 year
        key_minus = (name, birth_year - 1)
        if key_minus in tm_lookup:
            return tm_lookup[key_minus]
        
        key_plus = (name, birth_year + 1)
        if key_plus in tm_lookup:
            return tm_lookup[key_plus]
        
        return None
    
    master['player_id'] = master.apply(find_player_id, axis=1)
    
    exact_matches = master['player_id'].notna().sum()
    print(f"  ✓ Exact matches: {exact_matches:,}")
    
    # STAGE 2: Fuzzy matching (optimized - only for unmatched)
    unmatched = master[master['player_id'].isna()].copy()
    
    if len(unmatched) > 0:
        print(f"\nStage 2: Fuzzy matching for {len(unmatched):,} unmatched players...")
        
        # Pre-compute name words for all TM players
        tm_players['name_words'] = tm_players['name_clean'].apply(split_name_words)
        
        # Only search TM players with similar birth years (±3 years) for efficiency
        fuzzy_matches = []
        
        for idx, player_row in unmatched.iterrows():
            player_words = split_name_words(player_row['name_clean'])
            player_birth = player_row['birth_year_expected']
            
            if len(player_words) == 0:
                continue
            
            # Filter TM candidates by birth year first (±3 years max)
            if pd.notna(player_birth):
                candidates = tm_players[
                    (tm_players['birth_year'] >= player_birth - 3) &
                    (tm_players['birth_year'] <= player_birth + 3)
                ].copy()
            else:
                # No birth year - search all (slower)
                candidates = tm_players.copy()
            
            best_match = None
            best_score = 0
            
            for _, tm_row in candidates.iterrows():
                tm_words = tm_row['name_words']
                score = 0
                
                # Name word overlap
                name_overlap = len(player_words & tm_words)
                if name_overlap == 0:
                    continue
                score += name_overlap * 50
                
                # Birth year match
                if pd.notna(player_birth) and pd.notna(tm_row['birth_year']):
                    year_diff = abs(player_birth - tm_row['birth_year'])
                    if year_diff == 0:
                        score += 200
                    elif year_diff == 1:
                        score += 100
                    elif year_diff == 2:
                        score += 50
                
                # Market value bonus
                if pd.notna(tm_row['highest_market_value_in_eur']):
                    value = tm_row['highest_market_value_in_eur']
                    if value > 50000000:
                        score += 20
                    elif value > 10000000:
                        score += 15
                    elif value > 1000000:
                        score += 10
                
                if score > best_score and score >= 150:
                    best_score = score
                    best_match = tm_row
            
            if best_match is not None:
                fuzzy_matches.append({
                    'idx': idx,
                    'player_id': best_match['player_id'],
                    'master_name': player_row['name'],
                    'tm_name': best_match['name'],
                    'score': best_score
                })
        
        if fuzzy_matches:
            print(f"  ✓ Fuzzy matched: {len(fuzzy_matches)} additional players")
            
            # Apply matches
            for match in fuzzy_matches:
                master.loc[match['idx'], 'player_id'] = match['player_id']
    
    matched = master['player_id'].notna().sum()
    print(f"\n{'='*60}")
    print(f"✓ Total matched: {matched:,} / {len(master):,} ({matched/len(master)*100:.1f}%)")
    print(f"{'='*60}")
    
    # Clean up
    master = master.drop(columns=['birth_year_expected'], errors='ignore')
else:
    print("⚠️  Cannot match - missing data")

Matching players to Transfermarkt...



Stage 1: Exact name + birth year matching...
  ✓ Exact matches: 11,345

Stage 2: Fuzzy matching for 2,291 unmatched players...
  ✓ Fuzzy matched: 1623 additional players

✓ Total matched: 12,968 / 13,636 (95.1%)


## DEBUG: Check Unmatched Players

In [31]:
# Check players without Transfermarkt match
if 'master' in locals() and 'player_id' in master.columns:
    unmatched = master[master['player_id'].isna()].copy()
    
    print(f"Players without Transfermarkt match: {len(unmatched):,} rows\n")
    print(f"Unique unmatched players: {unmatched['name'].nunique()}\n")
    
    # Show sample
    print("Sample unmatched players (with details):")
    sample = unmatched[['name', 'Age', 'club', 'season', 'league_comp']].drop_duplicates('name').head(20)
    display(sample)
    
    # Show full list of unique names
    print(f"\nAll {unmatched['name'].nunique()} unmatched player names:")
    print(sorted(unmatched['name'].unique()))
    
    # Export to CSV for manual review if needed
    unmatched_summary = unmatched.groupby('name').agg({
        'Age': 'first',
        'club': 'first', 
        'season': 'first',
        'league_comp': 'first'
    }).reset_index()
    
    output_path = os.path.join('..', 'data', 'processed', 'unmatched_players.csv')
    unmatched_summary.to_csv(output_path, index=False)
    print(f"\n✓ Exported unmatched players to: {output_path}")
else:
    print("⚠️  player_id column not found")

Players without Transfermarkt match: 668 rows

Unique unmatched players: 412

Sample unmatched players (with details):


,name,Age,club,season,league_comp
42,Abass Issah,21.0,Mainz 05,2020-2021,de Bundesliga
57,Abdellah Baallal,18.0,Clermont Foot,2023-2024,fr Ligue 1
72,Abdoulay Diaby,29.0,Getafe,2020-2021,es La Liga
86,Abdoulaye Sylla,20.0,Nantes,2020-2021,fr Ligue 1
191,Admir Mehmedi,29.0,Wolfsburg,2020-2021,de Bundesliga
200,Adolfo Gaich,21.0,Benevento,2020-2021,it Serie A
207,Adrian Baquerin,17.0,Valladolid,2024-2025,es La Liga
257,Adryelson,25.0,Lyon,2023-2024,fr Ligue 1
273,Ahmed Touba,25.0,Lecce,2023-2024,it Serie A
274,Aiham Ousou,23.0,Cádiz,2023-2024,es La Liga



All 412 unmatched player names:
['Abass Issah', 'Abdellah Baallal', 'Abdoulay Diaby', 'Abdoulaye Sylla', 'Admir Mehmedi', 'Adolfo Gaich', 'Adrian Baquerin', 'Adryelson', 'Ahmed Touba', 'Aiham Ousou', 'Alejandro Rodriguez', 'Aleksander Buksa', 'Alessio Cacciamani', 'Alfreð Finnbogason', 'Alireza Jahanbakhsh', 'Andi Zeqiri', 'Andrew Gravillon', 'Andrey Lunyov', 'Andriy Yarmolenko', 'Andros Townsend', 'Ange Martial Tia', 'Antal Yaakobishvili', 'Ante Palaversa', 'Arber Zeneli', 'Arthur Cabral', 'Arturo Rodríguez', 'Asaf Arania', 'Asher Agbinone', 'Attila Szalai', 'Auston Trusty', 'Axel Tapé', 'Azzedine Ounahi', 'Babatunde Akinsola', 'Bakary Sako', 'Bartłomiej Drągowski', 'Bas Dost', 'Bastos', 'Beres Owusu', 'Berkan Kutlu', 'Bernard', 'Bernard Nguene', 'Bersant Celina', 'Bilal Boutobba', 'Birger Meling', 'Bobby Adekanye', 'Borja Sainz', 'Borja Vázquez', 'Bosko Sutalo', 'Bram Nuytinck', 'Brecht Dejaegere', 'Burak Yılmaz', 'Burgui', 'Béni Makouana', 'Cafú', 'Can Bozdogan', 'Cengiz Ünder', 'C

## 4. Get Market Values for Each Season

In [32]:
def get_market_value_for_season(player_id, season, tm_values_df):
    """
    Get market value at season start and end for a player.
    
    Returns: (value_at_start, value_at_end)
    """
    if pd.isna(player_id) or season not in SEASON_DATES:
        return None, None
    
    # Get this player's value history
    player_values = tm_values_df[tm_values_df['player_id'] == player_id].copy()
    
    if player_values.empty:
        return None, None
    
    player_values = player_values.sort_values('date')
    
    season_start = pd.to_datetime(SEASON_DATES[season]['start'])
    season_end = pd.to_datetime(SEASON_DATES[season]['end'])
    
    # Value at season start: most recent value before or at start date
    values_before_start = player_values[player_values['date'] <= season_start]
    if not values_before_start.empty:
        value_start = values_before_start.iloc[-1]['market_value_in_eur']
    else:
        # No value before season start, use first available
        value_start = player_values.iloc[0]['market_value_in_eur']
    
    # Value at season end: most recent value before or at end date
    values_before_end = player_values[player_values['date'] <= season_end]
    if not values_before_end.empty:
        value_end = values_before_end.iloc[-1]['market_value_in_eur']
    else:
        # No value before season end, use value_start
        value_end = value_start
    
    return value_start, value_end

print("✓ Market value function ready")

✓ Market value function ready


In [33]:
if not tm_values.empty and 'master' in locals() and 'player_id' in master.columns:
    print("Deduplicating before adding market values...\n")
    
    # Remove any duplicate player+season rows (keep first occurrence)
    before = len(master)
    master = master.drop_duplicates(subset=['name', 'season'], keep='first')
    after = len(master)
    
    if before != after:
        print(f"  Removed {before - after:,} duplicate player+season rows")
    
    print("\nGetting market values for each player-season...\n")
    
    # Apply function to each row
    market_values = master.apply(
        lambda row: get_market_value_for_season(row['player_id'], row['season'], tm_values),
        axis=1
    )
    
    # Split into two columns
    master['market_value_start'] = market_values.apply(lambda x: x[0])
    master['market_value_end'] = market_values.apply(lambda x: x[1])
    
    # Show coverage
    has_start = master['market_value_start'].notna().sum()
    has_end = master['market_value_end'].notna().sum()
    
    print(f"Market value coverage:")
    print(f"  Season start: {has_start:,} / {len(master):,} ({has_start/len(master)*100:.1f}%)")
    print(f"  Season end:   {has_end:,} / {len(master):,} ({has_end/len(master)*100:.1f}%)")
    
    print("\nCoverage by season:")
    display(master.groupby('season')['market_value_start'].agg(['count', lambda x: x.notna().sum()]))
else:
    print("⚠️  Cannot get market values - missing data")

Deduplicating before adding market values...

  Removed 31 duplicate player+season rows

Getting market values for each player-season...

Market value coverage:
  Season start: 12,938 / 13,605 (95.1%)
  Season end:   12,938 / 13,605 (95.1%)

Coverage by season:


,count,<lambda_0>
season,,
2020-2021,2507,2507
2021-2022,2621,2621
2022-2023,2590,2590
2023-2024,2612,2612
2024-2025,2608,2608


## 5. Preview Results

In [34]:
if 'master' in locals() and 'market_value_start' in master.columns:
    print("Sample players with market values:")
    sample = master[master['market_value_start'].notna()].head(10)
    display(sample[['name', 'club', 'season', 'market_value_start', 'market_value_end', 'goals', 'assists']])

Sample players with market values:


,name,club,season,market_value_start,market_value_end,goals,assists
0,Aaron Ciammaglichella,Torino,2024-2025,1000000.0,1000000.0,0,0
1,Aaron Connolly,Brighton,2020-2021,4000000.0,7000000.0,2,1
2,Aaron Connolly,Brighton,2021-2022,7000000.0,6000000.0,0,0
3,Aaron Cresswell,West Ham United,2020-2021,6500000.0,6000000.0,0,8
4,Aaron Cresswell,West Ham United,2021-2022,5000000.0,4000000.0,2,3
5,Aaron Cresswell,West Ham United,2022-2023,3000000.0,2500000.0,0,1
6,Aaron Cresswell,West Ham United,2023-2024,1200000.0,900000.0,0,0
7,Aaron Cresswell,West Ham United,2024-2025,900000.0,700000.0,0,0
8,Aaron Hickey,Bologna,2020-2021,600000.0,5000000.0,0,0
9,Aaron Hickey,Bologna,2021-2022,5000000.0,15000000.0,5,1


## 6. Save Updated Database

In [35]:
if 'master' in locals():
    # Drop temporary columns
    master = master.drop(columns=['name_clean', 'player_id'], errors='ignore')
    master = master.sort_values(['name', 'season']).reset_index(drop=True)
    
    # Save
    master.to_csv(DB_PATH, index=False)
    
    print("=" * 60)
    print(f"✓ SAVED: {DB_PATH}")
    print("=" * 60)
    print(f"Rows: {len(master):,}")
    print(f"Columns: {len(master.columns)}")
    print(f"\nNew columns: market_value_start, market_value_end")
else:
    print("⚠️  Nothing to save")

✓ SAVED: ..\data\processed\player_db.csv
Rows: 13,605
Columns: 38

New columns: market_value_start, market_value_end


## 7. Final Preview

In [36]:
if 'master' in locals():
    print("Final database structure:")
    master.info()
    print("\nSample:")
    display(master.head())

Final database structure:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13605 entries, 0 to 13604
Data columns (total 38 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   rank                13605 non-null  int64  
 1   name                13605 non-null  object 
 2   Nation              13600 non-null  object 
 3   position            13605 non-null  object 
 4   club                13605 non-null  object 
 5   league_comp         13605 non-null  object 
 6   Age                 13603 non-null  float64
 7   Born                13603 non-null  float64
 8   appearances         13605 non-null  int64  
 9   Starts              13605 non-null  int64  
 10  minutes             13605 non-null  int64  
 11  90s                 13605 non-null  float64
 12  goals               13605 non-null  int64  
 13  assists             13605 non-null  int64  
 14  G+A                 13605 non-null  int64  
 15  G-PK                13605 n

,rank,name,Nation,position,club,league_comp,Age,Born,appearances,Starts,...,xG,xA,xG90,xA90,xG90xA90,NPxG90xA90,xGChain90,xGBuildup90,market_value_start,market_value_end
0,2700,Aaron Ciammaglichella,it ITA,MF,Torino,it Serie A,19.0,2005.0,1,0,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1000000.0,1000000.0
1,1713,Aaron Connolly,ie IRL,FW/MF,Brighton,eng Premier League,20.0,2000.0,17,9,...,4.46,0.16,0.50,0.02,0.52,0.52,0.54,0.02,4000000.0,7000000.0
2,2318,Aaron Connolly,ie IRL,FW,Brighton,eng Premier League,21.0,2000.0,4,1,...,0.60,0.38,0.35,0.23,0.58,0.58,0.60,0.02,7000000.0,6000000.0
3,67,Aaron Cresswell,eng ENG,DF,West Ham United,eng Premier League,30.0,1989.0,36,36,...,0.88,7.39,0.03,0.21,0.23,0.23,0.30,0.24,6500000.0,6000000.0
4,269,Aaron Cresswell,eng ENG,DF,West Ham United,eng Premier League,31.0,1989.0,31,31,...,0.91,3.67,0.03,0.12,0.15,0.15,0.35,0.32,5000000.0,4000000.0
